# Week 1 · Tutorial 1 — What is data? Rows, sources, structure and variable types

**Covers videos 1.1 – 1.3:** observations and variables · structured vs. unstructured data · numerical vs. categorical variables.

**What you will do**
1. Ask *what one row is* and *who volunteered their data* (household survey).
2. Turn **unstructured text** (an outage log) into a table.
3. Decide **what to keep** when a time series is summarised.
4. Handle an **ordinal** variable (tariff bands) and an **identifier written as a number** (postcodes).

Each section gives you working code and one or more **✏️ Your turn** blocks. In those blocks you make a small,
clearly described change and run the cell. The numbers you get are what the quiz asks about, so keep the results.

> All data in this course is **simulated**, but it is built to look like real energy data.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/blasern/data-tutorials/main/data/"   # where the data files live
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

### A three-minute Python primer

You will meet the same few building blocks again and again. Run each cell below and read the output.

* **A variable** stores a value: `price = 45`. Text goes in quotes: `city = "Bergen"`.
* **A table** is called a *DataFrame* (`df`). A column is `df["column_name"]`; typical summaries are `.mean()`, `.max()`, `.sum()`, `.count()`.
* **A condition** gives True/False per row: `df["x"] > 3`. The mean of True/False values is a **share**: `(df["x"] > 3).mean()`.
* **Grouping**: `df.groupby("group")["x"].mean()` gives one mean per group.
* `round(value, 2)` rounds to 2 decimals. Anything after `#` is a comment and is ignored.

In [ ]:
price = 45                      # a variable
print("price is", price)

small = pd.DataFrame({"city": ["Bergen", "Oslo", "Bergen", "Tromso"], "kwh": [120, 340, 160, 80]})
print(small)
print("mean kwh:", small["kwh"].mean())                          # a column summary
print("share above 100 kWh:", (small["kwh"] > 100).mean())      # a condition -> share
print(small.groupby("city")["kwh"].mean())                      # one mean per group
print("rounded:", round(small["kwh"].mean(), 1))

## 1. A table is rows and columns — and someone chose who is in it

`households.csv` comes from a (fictional) utility. **One row is one household.**

| column | meaning |
|---|---|
| `household_id` | identifier |
| `postcode` | postcode of the household |
| `dwelling_type`, `heating` | categories |
| `floor_area_m2` | numeric |
| `tariff_band`, `tariff_band_code` | subscribed power band (text) and its code 1–4 |
| `annual_kwh` | metered consumption over one year |
| `in_app_panel` | 1 = household *volunteered* for the energy-saving app panel, 0 = did not |

In [ ]:
households = pd.read_csv(DATA_URL + "households.csv")
print(households.shape)
households.head()

The app company wants to say: *"The average household uses X kWh per year"*, using only its panel members.
First, how different is the panel from everybody else?

In [ ]:
print(households.groupby("in_app_panel")["annual_kwh"].agg(["count", "mean"]).round(0))

### ✏️ Your turn 1

The panel members volunteered. Check whether they are *typical* in more than just consumption.

Fill in the two `TODO` lines so that they compute **the share of households that heat with a `"heat pump"`**
(a number between 0 and 1, rounded to 3 decimals)
- `hp_share_panel`: among the panel members only (`in_app_panel == 1`)
- `hp_share_all`: among *all* households

Then run the cell and read the printed numbers.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
hp_share_panel = np.nan
hp_share_all = np.nan
```
by
```python
hp_share_panel = round((panel["heating"] == "heat pump").mean(), 3)
hp_share_all = round((households["heating"] == "heat pump").mean(), 3)
```

In [ ]:
panel = households[households["in_app_panel"] == 1]

hp_share_panel = np.nan   # TODO
hp_share_all = np.nan     # TODO

print("panel members:", len(panel))
print("heat-pump share, panel:", hp_share_panel)
print("heat-pump share, all  :", hp_share_all)
print("mean annual kWh, panel:", round(panel["annual_kwh"].mean()))
print("mean annual kWh, all  :", round(households["annual_kwh"].mean()))

*Think about it:* if the app company reported the panel mean as "the average household", which population would that
number actually describe? A smarter calculation cannot fix a biased sample.

## 2. Unstructured → structured: an outage log

Grid operators often keep **free-text logs**. `outage_log.txt` has one line per outage:

```
2023-01-03 20:28 | Feeder F13 tripped at Sotra. 169 customers off supply for 0h08m. Cause: overload.
```

The information is all there, but you cannot average a sentence. We use **regular expressions** to pull out the
parts we care about. Converting costs effort *and* forces us to choose what to keep.

In [ ]:
import urllib.request

with urllib.request.urlopen(DATA_URL + "outage_log.txt") as f:      # read the text file from the web
    lines = [line.decode("utf-8").strip() for line in f if line.strip()]
print(len(lines), "lines")
print(lines[0])

In [ ]:
import re

records = []
for line in lines:
    when = re.search(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2})", line).group(1)
    customers = int(re.search(r"(\d+) customers", line).group(1))
    h, m = re.search(r"for (\d+)h(\d+)m", line).groups()
    records.append({"start": pd.Timestamp(when), "customers": customers,
                    "duration_min": int(h) * 60 + int(m)})

outages = pd.DataFrame(records)
outages["customer_minutes"] = outages["customers"] * outages["duration_min"]
outages.head()

### ✏️ Your turn 2

The cause of each outage is still hidden in the text (`Cause: lightning.`).

1. Fill in the `TODO` so that the new column `cause` contains the cause as text (e.g. `"lightning"`,
   `"planned maintenance"`). Use the regular expression `r"Cause: ([a-z ]+)\."` and take group 1 — follow the
   pattern used for `customers` above.
2. Run the cell. It summarises the outages by cause.

**What to change** — edit the cell exactly like this, then run it:

Change `causes.append(None)` into `causes.append(re.search(r"Cause: ([a-z ]+)\.", line).group(1))`

In [ ]:
causes = []
for line in lines:
    causes.append(None)   # TODO: extract the cause with re.search(...)
outages["cause"] = causes

summary = (outages.groupby("cause", dropna=False)
           .agg(events=("customers", "size"),
                mean_duration_min=("duration_min", "mean"),
                customer_minutes=("customer_minutes", "sum"))
           .sort_values("customer_minutes", ascending=False))
print(summary.round(1))
print()
print("events longer than 3 hours:", (outages["duration_min"] > 180).sum())

## 3. Choose what you keep

Structured data can also be reduced. Here are 24 hours of 15-minute power readings (kW) from two households
(`two_households_15min.csv`).

In [ ]:
prof = pd.read_csv(DATA_URL + "two_households_15min.csv")
prof["hour"] = prof["time"].str[:2].astype(int)

summary = prof[["household_A_kw", "household_B_kw"]].agg(["mean", "std"]).round(3)
print(summary)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(prof.index / 4, prof["household_A_kw"], label="A")
ax.plot(prof.index / 4, prof["household_B_kw"], label="B")
ax.set_xlabel("hour of day"); ax.set_ylabel("kW"); ax.legend()
plt.show()

Mean and standard deviation are **identical** — yet the two households live very different lives. Which numbers
survive the conversion is a decision that needs domain knowledge.

### ✏️ Your turn 3

Add two features that *do* separate the households. Fill in the `TODO` lines:

- `peak_hour_*`: the **hour of day** (the integer in column `hour`) at which the household's highest 15-minute
  reading occurs. (Hint: `idxmax()` gives the row label of the maximum.)
- `evening_share_*`: the **share of the day's energy used between 17:00 and 21:59** (hours 17, 18, 19, 20, 21).
  Because all readings are equally spaced, this is simply the sum of readings in those hours divided by the sum of
  all readings. Round to 3 decimals.

Also compute `daily_kwh_*` = sum of readings × 0.25 h (each reading covers a quarter of an hour), rounded to 1 decimal.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
peak_hour_A = None
peak_hour_B = None
evening_share_A = np.nan
evening_share_B = np.nan
daily_kwh_A = np.nan
daily_kwh_B = np.nan
```
by
```python
peak_hour_A = int(prof.loc[prof["household_A_kw"].idxmax(), "hour"])
peak_hour_B = int(prof.loc[prof["household_B_kw"].idxmax(), "hour"])
evening_share_A = round(prof.loc[evening, "household_A_kw"].sum() / prof["household_A_kw"].sum(), 3)
evening_share_B = round(prof.loc[evening, "household_B_kw"].sum() / prof["household_B_kw"].sum(), 3)
daily_kwh_A = round(prof["household_A_kw"].sum() * 0.25, 1)
daily_kwh_B = round(prof["household_B_kw"].sum() * 0.25, 1)
```

In [ ]:
evening = prof["hour"].between(17, 21)

peak_hour_A = None       # TODO
peak_hour_B = None       # TODO
evening_share_A = np.nan # TODO
evening_share_B = np.nan # TODO
daily_kwh_A = np.nan     # TODO
daily_kwh_B = np.nan     # TODO

print("peak hour       A, B:", peak_hour_A, peak_hour_B)
print("evening share   A, B:", evening_share_A, evening_share_B)
print("daily energy    A, B:", daily_kwh_A, daily_kwh_B)

## 4. Numerical or categorical? What arithmetic is allowed

### 4a. An ordinal variable: tariff bands

`tariff_band` is an **ordered category**; `tariff_band_code` stores it as 1, 2, 3, 4. The codes look numerical, but the
*distance* between codes is not the same in kW: the bands have very different widths.

In [ ]:
print(households["tariff_band"].value_counts().reindex(["under 2 kW", "2-5 kW", "5-10 kW", "over 10 kW"]))
print()
print("mean of the codes:", round(households["tariff_band_code"].mean(), 3))

### ✏️ Your turn 4

The mean of a code has no unit. Replace it with something that does.

Use these band midpoints (in kW): `under 2 kW` → 1.0, `2-5 kW` → 3.5, `5-10 kW` → 7.5, `over 10 kW` → 13.0.

1. Fill in the `TODO`s: map each household's `tariff_band` to its midpoint (use `.map(midpoint)`), store it in
   the column `midpoint_kw`, and compute the mean of that column (rounded to 2 decimals).
2. Compute the share of households in `over 10 kW` (rounded to 3 decimals).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
households["midpoint_kw"] = np.nan
mean_midpoint = np.nan
share_over_10 = np.nan
```
by
```python
households["midpoint_kw"] = households["tariff_band"].map(midpoint)
mean_midpoint = round(households["midpoint_kw"].mean(), 2)
share_over_10 = round((households["tariff_band"] == "over 10 kW").mean(), 3)
```

In [ ]:
midpoint = {"under 2 kW": 1.0, "2-5 kW": 3.5, "5-10 kW": 7.5, "over 10 kW": 13.0}

households["midpoint_kw"] = np.nan    # TODO
mean_midpoint = np.nan                # TODO
share_over_10 = np.nan                # TODO

print("mean code        :", round(households["tariff_band_code"].mean(), 3))
print("mean midpoint kW :", mean_midpoint)
print("share over 10 kW :", share_over_10)

### 4b. An identifier that looks like a number: postcodes

`postcode` is stored as an integer. pandas happily computes its mean — but *postcode 5022 is not more than postcode 5008*.

In [ ]:
print("'mean postcode':", households["postcode"].mean().round(1), "  <- meaningless")
print(households["postcode"].dtype)

### ✏️ Your turn 5

Treat the postcode as what it is — a category.

1. Convert the column to text: `households["postcode"] = households["postcode"].astype(str)` (fill in the `TODO`).
2. Compute, for each postcode, the **number of households** and the **mean `annual_kwh`**, and store it in `by_pc`.
3. Print the number of distinct postcodes, the postcode with the highest mean consumption (and its mean, rounded to
   the nearest kWh), and how many postcodes have **fewer than 150 households**.

**What to change** — edit the cell exactly like this, then run it:

1. Change `# TODO: convert postcode to text` into `households["postcode"] = households["postcode"].astype(str)`

2. Replace
```python
by_pc = households.groupby("postcode")["annual_kwh"].agg(["count", "mean"])
```
by
```python
by_pc = households.groupby("postcode")["annual_kwh"].agg(["count", "mean"])
```

In [ ]:
# TODO: convert postcode to text

by_pc = households.groupby("postcode")["annual_kwh"].agg(["count", "mean"])   # works either way, but is only correct for a category

print("distinct postcodes:", households["postcode"].nunique())
print("highest mean consumption:", by_pc["mean"].idxmax(), round(by_pc["mean"].max()))
print("postcodes with < 150 households:", (by_pc["count"] < 150).sum())
print("dtype:", households["postcode"].dtype)

## Summary

* Before any analysis: **what does one row describe, how was the data generated, and which population does it represent?**
* Unstructured data can be converted, but you decide what to keep.
* Codes, identifiers and ordinal bands are not numbers — their arithmetic needs care.